# Neethi AI — Indian Kanoon SC Judgments Ingestion

> **Dataset:** [Supreme Court Judgments (India, 1950–2024)](https://www.kaggle.com/datasets/adarshsingh0903/legal-dataset-sc-judgments-india-19502024)  
> ~26,000 PDF judgments scraped from Indian Kanoon

---

## Actual PDF Structure (verified from real sample)

```
Every Page
  HEADER (y < 6%)  : "[Case] vs [Case] ... on [Date]"  ← repeats on every page, STRIP
  BODY             : Judgment text
  FOOTER (y > 92%) : "Indian Kanoon - http://indiankanoon.org/doc/{ID}/\n{page}\n"
                     ← URL here (text, NOT hyperlink annotation), STRIP after extracting URL

Page 1 Special Blocks (y < 170)
  Title block  : "Petitioner vs Respondent ... on DD Month, YYYY"  (multi-line)
  Author block : "Author: Judge Name"                              ← explicit field
  Bench block  : "Bench: Judge1, Judge2"                          ← explicit field
  Court block  : "REPORTABLE\nIN THE SUPREME COURT OF INDIA\nCRIMINAL..."
  Party block  : "PETITIONER NAME   ...APPELLANT"
  Party block  : "RESPONDENT NAME   ...RESPONDENT"
  Case num     : "CRIMINAL APPEAL NO.172 OF 2023"
```

**Key finding:** The Indian Kanoon URL lives in the footer text — NOT in PDF hyperlink annotations (get_links() returns empty). It must be extracted from the footer block before stripping.

---

## Kaggle Setup (Before Running)

| Setting | Value |
|---------|-------|
| **Accelerator** | T4 GPU x2 or P100 |
| **Internet** | ON — needed for Qdrant API |
| **Dataset** | Add `adarshsingh0903/legal-dataset-sc-judgments-india-19502024` |

> **Time estimate:** ~2–3 h for all 26K PDFs with extractive summarization on T4 GPU.

In [ ]:
# Install dependencies
# Kaggle has torch, transformers, and huggingface_hub pre-installed.
# FlagEmbedding is NOT used — BGE-M3 is loaded directly via transformers
# (FlagEmbedding has Python 3.12 incompatibilities in BGE_M3/modeling.py).
!pip install -q \
    'qdrant-client>=1.12.0' \
    'pymupdf>=1.24.0' \
    'pdfplumber>=0.11.0' \
    'mistralai>=1.0.0' \
    'tqdm'

import torch
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
import os, re, json, uuid, time, logging, warnings
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import fitz           # PyMuPDF
import pdfplumber
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download

from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance,
    SparseVectorParams, SparseIndexParams,
    ScalarQuantization, ScalarQuantizationConfig, ScalarType,
    HnswConfigDiff, OptimizersConfigDiff,
    PointStruct, SparseVector,
    Filter, FieldCondition, MatchValue, Range,
)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)-8s %(message)s')
logger = logging.getLogger('neethi_ingest')

import importlib.metadata
print(f'qdrant-client : {importlib.metadata.version("qdrant-client")}')
print(f'transformers  : {importlib.metadata.version("transformers")}')
print(f'torch         : {torch.__version__}')
print('Imports OK')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit this cell before running          ║
# ╚══════════════════════════════════════════════════════════╝

# Qdrant
QDRANT_URL      = 'https://YOUR-CLUSTER.qdrant.io:6333'   # <- replace
QDRANT_API_KEY  = 'YOUR-QDRANT-API-KEY'                  # <- replace
COLLECTION_NAME = 'indian_kanoon'

# Embedding model
EMBEDDING_MODEL      = 'BAAI/bge-m3'   # 1024-dim dense + sparse BM25
EMBEDDING_BATCH_SIZE = 32              # lower to 16 if GPU OOM
USE_FP16             = True
USE_SPARSE           = True            # store BM25 sparse vectors
DENSE_DIM            = 1024

# LLM (Mistral Large)
# 'none'    = fast extractive summaries only, no API calls
# 'mistral' = LLM-based summaries + LLM verdict fallback (requires API key)
LLM_PROVIDER    = 'none'
MISTRAL_API_KEY = ''                       # <- replace with your Mistral API key
MISTRAL_MODEL   = 'mistral-large-latest'
LLM_RPM_DELAY   = 0.5                     # seconds between Mistral requests

# ── Paths ──────────────────────────────────────────────────────────────
# The dataset is organised as:
#   {JUDGMENTS_DIR}/{year}/{case_name}.PDF
#
# Example:
#   .../supreme_court_judgments/1950/A_K_Gopalan_vs_The_State_Of_Madras_on_19_May_1950_1.PDF
#   .../supreme_court_judgments/2023/Ajay_Kumar_Goenka_vs_Tourism_Finance_on_15_March_2023_1.PDF
JUDGMENTS_DIR   = '/kaggle/input/datasets/adarshsingh0903/legal-dataset-sc-judgments-india-19502024/supreme_court_judgments'
CHECKPOINT_FILE = '/kaggle/working/ik_checkpoint.json'
ERROR_LOG       = '/kaggle/working/ik_errors.jsonl'

# ── Year filter ─────────────────────────────────────────────────────────
# Only ingest PDFs from year directories >= START_YEAR.
# The year is read directly from the directory name (reliable, no regex needed).
# 2000–2024 ≈ 14,000–16,000 PDFs  |  1950–2024 ≈ 26,000 PDFs
START_YEAR = 2000   # change to 1950 to ingest everything

# ── Fresh start flag ────────────────────────────────────────────────────
# FRESH_START = True  → deletes the existing Qdrant collection and clears
#                       the checkpoint so every PDF is re-processed from
#                       scratch.  Set to False to resume a previous run.
# FRESH_START = False → resume mode: already-processed files are skipped
#                       and the collection is left intact.
FRESH_START = True   # <- set to False to resume a previous run

# Processing
BATCH_SIZE     = 64    # Qdrant upsert batch size
WORKER_THREADS = 4     # parallel PDF-parsing threads
MAX_TEXT_STORE = 12000 # max chars of clean text kept in payload
PREVIEW_CHARS  = 1800  # chars for searchable text_preview field
MAX_DOCS       = None  # set to int to cap (e.g. 100 for a quick test)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Quick sanity check on paths
from pathlib import Path
judgments_path = Path(JUDGMENTS_DIR)
print(f'Judgments dir exists : {judgments_path.exists()}')
if judgments_path.exists():
    year_dirs = sorted(
        [d for d in judgments_path.iterdir() if d.is_dir() and d.name.isdigit()],
        key=lambda d: int(d.name)
    )
    all_years  = [int(d.name) for d in year_dirs]
    filt_years = [y for y in all_years if y >= START_YEAR]
    sample_pdfs = list(year_dirs[-1].glob('*.PDF')) + list(year_dirs[-1].glob('*.pdf'))

    yr_min = str(min(all_years))  if all_years  else 'none'
    yr_max = str(max(all_years))  if all_years  else 'none'
    fy_min = str(min(filt_years)) if filt_years else 'none'
    fy_max = str(max(filt_years)) if filt_years else 'none'

    print(f'Year dirs total      : {len(all_years)}  ({yr_min}-{yr_max})')
    print(f'Year dirs >= {START_YEAR}  : {len(filt_years)}  ({fy_min}-{fy_max})')
    if sample_pdfs:
        print(f'Sample PDF path      : {sample_pdfs[0]}')
else:
    print('  -> Check that the dataset is added as a Kaggle notebook input')

print(f'\nEmbedding model      : {EMBEDDING_MODEL}')
print(f'Sparse vectors       : {USE_SPARSE}')
print(f'LLM provider         : {LLM_PROVIDER}')
print(f'Device               : {DEVICE}')
print(f'Start year filter    : >= {START_YEAR}')
print(f'Fresh start          : {FRESH_START}')


In [ ]:
# Qdrant — connect and create collection
#
# Memory design for free tier (1 GB RAM):
#   INT8 scalar quantization  → 4x smaller dense vectors
#   One vector per document   → 26K points = ~26 MB vectors
#   Payload ~6 KB/doc         → ~156 MB payload
#   Total                     → well under 1 GB
#
# FRESH_START=True  → existing collection is deleted and recreated from scratch.
# FRESH_START=False → existing collection is reused (resume mode).

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

def _build_collection(client: QdrantClient, name: str) -> None:
    """Create the collection with INT8 quantization and sparse vectors."""
    client.create_collection(
        collection_name=name,
        vectors_config={
            'dense': VectorParams(
                size=DENSE_DIM,
                distance=Distance.COSINE,
                quantization_config=ScalarQuantization(
                    scalar=ScalarQuantizationConfig(
                        type=ScalarType.INT8,
                        quantile=0.99,
                        always_ram=True,
                    )
                ),
                hnsw_config=HnswConfigDiff(
                    m=16,
                    ef_construct=100,
                    on_disk=False,
                ),
            )
        },
        sparse_vectors_config={
            'sparse': SparseVectorParams(
                index=SparseIndexParams(on_disk=False, full_scan_threshold=5000)
            )
        } if USE_SPARSE else None,
        optimizers_config=OptimizersConfigDiff(
            indexing_threshold=10000,
            memmap_threshold=50000,
        ),
        on_disk_payload=False,
    )

    # Payload indexes for fast filtering
    for field, schema in [
        ('year',         'integer'),
        ('verdict_type', 'keyword'),
        ('case_type',    'keyword'),
        ('has_url',      'bool'),
        ('summary_type', 'keyword'),
    ]:
        client.create_payload_index(
            collection_name=name,
            field_name=field,
            field_schema=schema,
        )

    print(f"Created '{name}' — INT8 quantization, sparse={'yes' if USE_SPARSE else 'no'}")


def setup_collection(client: QdrantClient, name: str, fresh: bool) -> None:
    existing = {c.name for c in client.get_collections().collections}

    if name in existing:
        if fresh:
            existing_ct = client.count(name).count
            print(f"FRESH_START=True — deleting '{name}' ({existing_ct:,} points) ...")
            client.delete_collection(name)
            print(f"  Deleted. Recreating ...")
            _build_collection(client, name)
        else:
            ct = client.count(name).count
            print(f"Collection '{name}' exists — {ct:,} points (resume mode).")
    else:
        _build_collection(client, name)


setup_collection(qdrant, COLLECTION_NAME, fresh=FRESH_START)


In [ ]:
# PDF Parser
#
# Real PDF structure (verified from actual Indian Kanoon PDFs):
#
#  Every page
#    HEADER (y < ~6% of page height)
#      "Petitioner vs Respondent ... on Date"  -- same line on every page
#    FOOTER (y > ~92% of page height)
#      "Indian Kanoon - http://indiankanoon.org/doc/{ID}/\n{page_num}\n"
#      NOTE: URL is text only -- get_links() returns EMPTY on these PDFs!
#
#  Page 1 special blocks (y < ~170 px from top, after header)
#    Title block  : "Petitioner vs Respondent ... on DD Month, YYYY"
#    Author block : "Author: Judge Name"       <- explicit structured field
#    Bench block  : "Bench: Judge1, Judge2"    <- explicit structured field
#    Court block  : "REPORTABLE\nIN THE SUPREME COURT OF INDIA\n..."
#    Party blocks : "PETITIONER NAME  ...APPELLANT"
#                   "RESPONDENT NAME  ...RESPONDENT"
#    Case num     : "CRIMINAL APPEAL NO.172 OF 2023"
#
# Strategy:
#  Pass 1  -- scan every page for footer URL and page-1 structured metadata
#  Pass 2  -- extract clean body text (header + footer stripped)

class SCJudgmentParser:

    # Matches IK footer URL from text
    _IK_URL_RE = re.compile(
        r'(?:Indian\s+Kanoon\s*[-\u2013]?\s*)'
        r'(https?://(?:www\.)?indiankanoon\.org/doc/(\d+)/?)'
    )
    _IK_URL_BARE = re.compile(
        r'https?://(?:www\.)?indiankanoon\.org/doc/(\d+)/?'
    )

    # Line-level noise (applied after position stripping)
    _NOISE = [
        re.compile(r'^\s*\d{1,4}\s*$'),                           # bare page number
        re.compile(r'^\s*[-\u2013\u2014]\s*\d+\s*[-\u2013\u2014]\s*$'),  # - 3 -
        re.compile(r'^\s*Page\s+\d+\s+of\s+\d+\s*$', re.I),
        re.compile(r'^\s*Indian\s+Kanoon.*indiankanoon.*$', re.I),
        re.compile(r'^\s*https?://(?:www\.)?indiankanoon.*$', re.I),
        re.compile(r'^\s*www\.indiankanoon\.org\s*$', re.I),
        re.compile(r'^\s*REPORTABLE\s*$', re.I),
        re.compile(r'^\s*NOT\s+REPORTABLE\s*$', re.I),
        re.compile(r'^\s*NON-REPORTABLE\s*$', re.I),
        re.compile(r'^\s*\*{3,}\s*$'),
        re.compile(r'^\s*Xxx\s+xxx\s+xxx\s*$', re.I),
    ]

    def parse(self, pdf_path: str) -> Optional[Dict[str, Any]]:
        """Return dict: clean_text, structured_meta, ik_url, page_count."""
        try:
            return self._parse_pymupdf(pdf_path)
        except Exception:
            try:
                return self._parse_pdfplumber(pdf_path)
            except Exception as exc:
                logger.warning('Both parsers failed %s: %s', pdf_path, exc)
                return None

    # ── PyMuPDF (primary) ─────────────────────────────────────────────

    def _parse_pymupdf(self, path: str) -> Dict[str, Any]:
        doc = fitz.open(path)
        n_pages = len(doc)

        ik_url         = None
        structured     = {}   # author, bench, title, petitioner, respondent, case_number
        body_pages     = []   # clean text per page

        for page_num, page in enumerate(doc):
            ph = page.rect.height
            top_clip    = ph * 0.065   # header zone upper boundary
            bottom_clip = ph * 0.920   # footer zone lower boundary

            page_body_lines = []

            for blk in page.get_text('blocks', sort=True):
                x0, y0, x1, y1, text, _, btype = blk
                if btype == 1:   # skip image blocks
                    continue
                text = text.strip()
                if not text:
                    continue

                mid_y = (y0 + y1) / 2

                # ── Footer zone: extract URL, then skip ───────────────
                if mid_y > bottom_clip:
                    if ik_url is None:
                        m = self._IK_URL_RE.search(text)
                        if m:
                            ik_url = f'https://indiankanoon.org/doc/{m.group(2)}'
                        else:
                            m2 = self._IK_URL_BARE.search(text)
                            if m2:
                                ik_url = f'https://indiankanoon.org/doc/{m2.group(1)}'
                    continue   # always skip footer from body text

                # ── Header zone: skip ─────────────────────────────────
                if mid_y < top_clip:
                    continue

                # ── Page 1 structured metadata (y < 170) ─────────────
                # These blocks contain Author / Bench / Title / Party info.
                # We extract them here AND also include them in body text
                # so the full judgment is searchable.
                if page_num == 0 and y1 < 170:
                    line = text.strip()
                    if line.startswith('Author:'):
                        structured['author'] = line[7:].strip()
                        continue     # author line not useful in body
                    if line.startswith('Bench:'):
                        structured['bench_raw'] = line[6:].strip()
                        continue     # bench line not useful in body
                    # Title block: "X vs Y ... on Date" (multi-line)
                    if 'vs' in line.lower() or 'versus' in line.lower():
                        title_candidate = re.sub(r'\s+', ' ', line)
                        if len(title_candidate) > 20:
                            structured.setdefault('title_block', title_candidate)
                        continue

                # ── Party lines on page 1 ────────────────────────────
                if page_num == 0 and y1 < 380:
                    upper = text.upper()
                    if 'APPELLANT' in upper or 'PETITIONER' in upper:
                        # Party name precedes ...APPELLANT
                        party = re.split(r'\.{2,}|\s{3,}|[\u2026]', text)[0].strip()
                        party = re.sub(r'[\ufffd\u0092\u2019]', '', party).strip()
                        if 3 < len(party) < 200:
                            structured.setdefault('petitioner', party)
                    elif 'RESPONDENT' in upper or 'DEFENDANT' in upper:
                        party = re.split(r'\.{2,}|\s{3,}|[\u2026]', text)[0].strip()
                        party = re.sub(r'[\ufffd\u0092\u2019]', '', party).strip()
                        if 3 < len(party) < 200:
                            structured.setdefault('respondent', party)

                # ── Body text ────────────────────────────────────────
                page_body_lines.append(text)

            body_pages.append('\n'.join(page_body_lines))

        doc.close()

        raw_body = '\n'.join(body_pages)
        clean    = self._clean(raw_body)

        # Fallback: scan body for URL if footer scan missed it
        if ik_url is None:
            m = self._IK_URL_BARE.search(raw_body)
            if m:
                ik_url = f'https://indiankanoon.org/doc/{m.group(1)}'

        return {
            'clean_text':    clean,
            'structured':    structured,
            'ik_url':        ik_url,
            'page_count':    n_pages,
        }

    # ── pdfplumber (fallback) ─────────────────────────────────────────

    def _parse_pdfplumber(self, path: str) -> Dict[str, Any]:
        pages = []
        ik_url = None
        with pdfplumber.open(path) as pdf:
            n = len(pdf.pages)
            for pg in pdf.pages:
                w, h = pg.width, pg.height
                # Extract footer for URL before stripping
                footer = pg.crop((0, h * 0.92, w, h))
                footer_text = footer.extract_text() or ''
                if ik_url is None:
                    m = self._IK_URL_BARE.search(footer_text)
                    if m:
                        ik_url = f'https://indiankanoon.org/doc/{m.group(1)}'
                # Body text
                body = pg.crop((0, h * 0.065, w, h * 0.92))
                pages.append(body.extract_text() or '')

        raw = '\n'.join(pages)
        # Extract author/bench from raw text
        structured = self._extract_structured_from_text(raw)
        return {
            'clean_text': self._clean(raw),
            'structured': structured,
            'ik_url':     ik_url,
            'page_count': n,
        }

    # ── Text cleaning ─────────────────────────────────────────────────

    def _clean(self, text: str) -> str:
        lines = text.split('\n')
        kept = []
        for line in lines:
            s = line.strip()
            if not s:
                continue
            if any(p.match(s) for p in self._NOISE):
                continue
            kept.append(s)
        out = '\n'.join(kept)
        # Collapse repeated court headers
        out = re.sub(
            r'(?:IN THE SUPREME COURT OF INDIA\s*\n)+',
            'IN THE SUPREME COURT OF INDIA\n',
            out, flags=re.I
        )
        # Remove garbled Unicode artifacts common in Indian legal PDFs
        out = re.sub(r'[\ufffd\x92\x93\x94]+', "'", out)
        out = re.sub(r'\n{3,}', '\n\n', out)
        return out.strip()

    # ── Extract Author/Bench from raw text (pdfplumber fallback) ──────

    def _extract_structured_from_text(self, text: str) -> Dict:
        structured = {}
        for line in text.split('\n')[:30]:
            ls = line.strip()
            if ls.startswith('Author:'):
                structured['author'] = ls[7:].strip()
            elif ls.startswith('Bench:'):
                structured['bench_raw'] = ls[6:].strip()
        return structured


_parser = SCJudgmentParser()
print('SCJudgmentParser ready')

In [ ]:
# Quick smoke test — run on any PDF you can access locally
# In the Kaggle notebook change this to a path under /kaggle/input/
#
# Uncomment to test:
#
# TEST_PDF = '/kaggle/input/legal-dataset-sc-judgments-india-19502024/some_judgment.pdf'
# result = _parser.parse(TEST_PDF)
# if result:
#     print('IK URL      :', result['ik_url'])
#     print('Structured  :', json.dumps(result['structured'], indent=2))
#     print('Text (first 500 chars):')
#     print(result['clean_text'][:500])
#     print('...')
#     print('Text (last 300 chars):')
#     print(result['clean_text'][-300:])
print('Smoke test cell — uncomment TEST_PDF to verify your PDFs')

In [ ]:
# Metadata Extractor
#
# Verified extraction logic based on actual PDF analysis:
#
# ┌─────────────────┬──────────────────────────────────────────────────┐
# │ Field           │ Source                                           │
# ├─────────────────┼──────────────────────────────────────────────────┤
# │ case_title      │ structured['title_block'] from page-1 body block │
# │ author          │ structured['author'] from "Author: ..." line     │
# │ judges          │ structured['bench_raw'] from "Bench: ..." line   │
# │ petitioner      │ structured['petitioner'] from party block        │
# │ respondent      │ structured['respondent'] from party block        │
# │ case_number     │ regex from "CRIMINAL APPEAL NO.172 OF 2023"      │
# │ date            │ regex from title "... on DD Month, YYYY"         │
# │ year            │ from date or citations                           │
# │ citations       │ SCC / AIR / SCC OnLine patterns                  │
# │ legal_sections  │ Section X of [Act] / Article X of Constitution  │
# │ verdict_type    │ regex (last 20% of text) → Mistral fallback      │
# │ case_type       │ regex from appeal/petition type                  │
# └─────────────────┴──────────────────────────────────────────────────┘

# ── Mistral verdict fallback ──────────────────────────────────────────
# Called only when all regex patterns fail to classify the verdict.
# Uses the last 800 chars (operative paragraph territory) and asks
# Mistral Large to classify with a single constrained word.
# Returns 'unknown' on any API/network error — ingestion always continues.

_VALID_VERDICTS = frozenset(
    ['allowed', 'dismissed', 'partly_allowed', 'disposed',
     'remanded', 'set_aside', 'withdrawn', 'unknown']
)

def _mistral_verdict(tail: str) -> str:
    """LLM fallback — classify verdict when regex fails."""
    if not MISTRAL_API_KEY:
        return 'unknown'
    try:
        from mistralai import Mistral
        _client = Mistral(api_key=MISTRAL_API_KEY)
        resp = _client.chat.complete(
            model=MISTRAL_MODEL,
            messages=[
                {
                    'role': 'user',
                    'content': (
                        'This is the closing section of an Indian Supreme Court judgment:\n\n'
                        f'{tail[-800:]}\n\n'
                        'What is the final verdict? Reply with EXACTLY ONE word from this list:\n'
                        'allowed, dismissed, partly_allowed, disposed, remanded, set_aside, withdrawn, unknown'
                    ),
                }
            ],
            max_tokens=8,
            temperature=0.0,
        )
        word = resp.choices[0].message.content.strip().lower()
        word = re.sub(r'[^a-z_]', '', word)
        if word in _VALID_VERDICTS:
            return word
    except Exception as exc:
        logger.debug('Mistral verdict fallback failed: %s', exc)
    return 'unknown'


class MetadataExtractor:

    # ── Citations ─────────────────────────────────────────────────────
    _CITE_FULL = [
        re.compile(r'\(\d{4}\)\s*\d+\s*SCC\s*\d+(?:\s*\(\w+\))?'),
        re.compile(r'AIR\s*\d{4}\s*SC\s*\d+'),
        re.compile(r'\d{4}\s*SCC\s*OnLine\s*SC\s*\d+'),
        re.compile(r'MANU/SC/\d+/\d{4}'),
    ]
    _CITE_YEAR = [
        re.compile(r'\((\d{4})\)\s*\d+\s*SCC'),
        re.compile(r'AIR\s*(\d{4})\s*SC'),
        re.compile(r'MANU/SC/\d+/(\d{4})'),
        re.compile(r'(\d{4})\s*SCC\s*OnLine'),
    ]

    # ── Date from title line: "... on 15 March, 2023" ─────────────────
    _DATE_FROM_TITLE = re.compile(
        r'\bon\s+(\d{1,2}(?:st|nd|rd|th)?\s+'
        r'(?:January|February|March|April|May|June|July|August'
        r'|September|October|November|December),?\s+\d{4})\s*$',
        re.I
    )
    _DATE_DOC_END = re.compile(
        r'\b((?:JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST'
        r'|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+\d{1,2},?\s+\d{4})',
        re.I
    )
    _DATE_GENERAL = re.compile(
        r'(\d{1,2}(?:st|nd|rd|th)?\s+'
        r'(?:January|February|March|April|May|June|July|August'
        r'|September|October|November|December),?\s+\d{4})',
        re.I
    )

    # ── Case number ──────────────────────────────────────────────────
    _CASE_NUM = re.compile(
        r'((?:CRIMINAL|CIVIL|WRIT|TRANSFER|REVIEW|CURATIVE|ELECTION)\s+'
        r'(?:APPEAL|PETITION|SUIT)\s*(?:NO\.?|NUMBER)?\s*[\d]+\s*(?:OF\s*\d{4})?|'
        r'(?:Crl\.?|Civil)\.?\s*A\.?\s*No\.?\s*[\d/]+(?:\s*of\s*\d{4})?|'
        r'SLP\s*(?:\(Crl\.?\)|\(C\))?\s*No\.?\s*[\d/]+(?:\s*of\s*\d{4})?)',
        re.I
    )

    # ── Legal sections ────────────────────────────────────────────────
    _SECTIONS = [
        re.compile(
            r'[Ss]ection[s]?\s+(\d+[A-Z]?(?:\(\d+\))?(?:\([a-z]\))?)'
            r'(?:-[A-Z])?\s+(?:of\s+)?(?:the\s+)?'
            r'([A-Z][A-Za-z\s,]+(?:Act|Code|Sanhita|Adhiniyam)[\s,\d]*)'
        ),
        re.compile(
            r'[Ss]ection[s]?\s+(\d+[A-Z]?(?:-[A-Z])?)\s+'
            r'(IPC|CrPC|CPC|IEA|BNS|BNSS|BSA|NI\s*Act|IBC|IT\s*Act|PMLA)'
        ),
        re.compile(
            r'\b(IPC|CrPC|CPC|IEA|BNS|BNSS|BSA|NI\s*Act|IBC)\s+'
            r'[Ss](?:ection)?\.?\s*(\d+[A-Z]?(?:-[A-Z])?)'
        ),
        re.compile(
            r'[Aa]rticle[s]?\s+(\d+[A-Z]?(?:\(\d+\))?(?:\([a-z]\))?)'
            r'\s+(?:of\s+)?(?:the\s+)?Constitution'
        ),
    ]

    # ── Verdict (regex tier) ──────────────────────────────────────────
    # Covers the full range of SC operative paragraph styles.
    # When all patterns fail, _mistral_verdict() is called as fallback.
    _VERDICT = [
        # "is/are hereby allowed/dismissed/disposed of/remanded"
        re.compile(
            r'(?:the\s+)?(?:these\s+)?'
            r'(?:appeal|petition|application|suit|matter|writ|revision)s?\s+'
            r'(?:fail[s]?\s+and\s+(?:is|are)\s+|(?:is|are)\s+hereby\s+)'
            r'(allowed|dismissed|partly\s+allowed|disposed\s+of|withdrawn|remanded)',
            re.I
        ),
        # "the/these appeal/petition is/are allowed/dismissed" (without "hereby")
        re.compile(
            r'(?:the\s+|these\s+)?'
            r'(?:appeal|petition|application|writ|matter|revision)s?\s+'
            r'(?:is|are)\s+(allowed|dismissed|partly\s+allowed|disposed\s+of|remanded)',
            re.I
        ),
        # "stands/stand allowed/dismissed"
        re.compile(
            r'(?:appeal|petition|application|matter)s?\s+(?:stands?|stand)\s+'
            r'(allowed|dismissed|disposed\s+of|withdrawn)',
            re.I
        ),
        # "we/this court allow/dismiss/set aside/quash ..."
        re.compile(
            r'(?:we|this\s+court)\s+(?:hereby\s+)?'
            r'(allow|dismiss|set\s+aside|quash|uphold|affirm)',
            re.I
        ),
        # "accordingly/therefore/in the result ... allowed/dismissed"
        re.compile(
            r'(?:accordingly|therefore|thus|hence|in\s+(?:the\s+)?result'
            r'|for\s+(?:the\s+)?(?:above|foregoing|aforesaid|these)\s+reasons?'
            r'|in\s+view\s+of\s+the\s+(?:above|foregoing))\s*[,.]?\s*'
            r'(?:(?:the\s+|these\s+)?(?:appeal|petition|application|matter)s?\s+)?'
            r'(?:(?:is|are|stands?)\s+)?(allowed|dismissed|disposed\s+of|partly\s+allowed)',
            re.I
        ),
        # "dismissed/allowed with/without costs"
        re.compile(
            r'\b(allowed|dismissed|disposed\s+of)\s+(?:with|without)\s+(?:any\s+)?costs?',
            re.I
        ),
        # "the impugned order/judgment is set aside / affirmed / upheld"
        re.compile(
            r'(?:the\s+)?impugned\s+(?:order|judgment|decision|award|conviction)\s+'
            r'(?:is|are|stands?)\s+(set\s+aside|quashed|affirmed|upheld|modified)',
            re.I
        ),
        # "the appeal fails" (no explicit "dismissed")
        re.compile(
            r'(?:the\s+)?(?:these\s+)?(?:appeal|petition|application)s?\s+fail',
            re.I
        ),
        # "no merit in the appeal/petition" → dismissed
        re.compile(
            r'(?:find|found|see|have|having)\s+no\s+merit\s+in\s+'
            r'(?:the|these|this)\s+(?:appeal|petition|application)',
            re.I
        ),
    ]

    # ── Case type ─────────────────────────────────────────────────────
    _CASE_TYPE = re.compile(
        r'\b(CRIMINAL\s+APPEAL|CIVIL\s+APPEAL|WRIT\s+PETITION'
        r'|SPECIAL\s+LEAVE\s+PETITION|\bSLP\b|TRANSFER\s+PETITION'
        r'|CURATIVE\s+PETITION|REVIEW\s+PETITION|ELECTION\s+PETITION)\b',
        re.I,
    )

    # ─────────────────────────────────────────────────────────────────

    def extract(self, clean_text: str, structured: Dict, file_name: str) -> Dict[str, Any]:
        """Extract all metadata. structured = page-1 blocks from parser."""
        head = clean_text[:5000]
        # Last 20% for verdict (operative paragraph is always near the end)
        tail = clean_text[int(len(clean_text) * 0.80):]

        meta = {
            'case_title':       self._title(structured, clean_text, file_name),
            'author':           structured.get('author', ''),
            'judges':           self._judges(structured),
            'petitioner':       self._clean_party(structured.get('petitioner', '')),
            'respondent':       self._clean_party(structured.get('respondent', '')),
            'case_number':      self._case_number(head),
            'citations':        self._citations(head),
            'primary_citation': self._primary_citation(head),
            'date':             self._date(structured, clean_text),
            'year':             self._year(structured, clean_text),
            'legal_sections':   self._sections(clean_text),
            'verdict_type':     self._verdict(tail),
            'case_type':        self._case_type(head),
        }
        return meta

    def _title(self, structured: Dict, text: str, file_name: str) -> str:
        if structured.get('title_block'):
            t = re.sub(r'\s+', ' ', structured['title_block'])
            t = re.sub(r'\s+\.{3,}\s+on\s+.+$', '', t, flags=re.I)
            t = re.sub(r'\s+on\s+\d{1,2}\s+\w+,?\s+\d{4}\s*$', '', t, flags=re.I)
            if len(t) > 10:
                return t.strip()[:300]
        for line in text.split('\n')[:25]:
            ls = line.strip()
            if re.search(r'\bvs?\.?\b|\bversus\b', ls, re.I) and 10 < len(ls) < 350:
                return ls
        return Path(file_name).stem.replace('_', ' ')[:300]

    def _judges(self, structured: Dict) -> List[str]:
        bench = structured.get('bench_raw', '')
        if bench:
            names = re.split(r',\s*|\s+and\s+|\s+&\s+', bench, flags=re.I)
            cleaned = [n.strip() for n in names if 3 < len(n.strip()) < 80]
            if cleaned:
                return cleaned[:5]
        author = structured.get('author', '')
        if author:
            return [author]
        return []

    def _clean_party(self, party: str) -> str:
        if not party:
            return ''
        party = re.sub(r'[\ufffd\u0092\u2019\u2026]+', '', party)
        party = re.sub(r'\s+', ' ', party).strip()
        return party[:200]

    def _case_number(self, text: str) -> str:
        m = self._CASE_NUM.search(text)
        if m:
            return re.sub(r'\s+', ' ', m.group(0).strip())
        return ''

    def _citations(self, text: str) -> List[str]:
        found = []
        for pat in self._CITE_FULL:
            found += pat.findall(text)
        return list(dict.fromkeys(found))[:10]

    def _primary_citation(self, text: str) -> str:
        for pat in self._CITE_FULL:
            m = pat.search(text)
            if m:
                return m.group(0).strip()
        return ''

    def _date(self, structured: Dict, text: str) -> str:
        title_blk = structured.get('title_block', '')
        m = self._DATE_FROM_TITLE.search(title_blk)
        if m:
            return m.group(1).strip()
        tail = text[-2000:]
        m = self._DATE_DOC_END.search(tail)
        if m:
            return m.group(1).strip()
        m = self._DATE_GENERAL.search(text[:3000])
        if m:
            return m.group(1).strip()
        return ''

    def _year(self, structured: Dict, text: str) -> Optional[int]:
        title_blk = structured.get('title_block', '')
        ym = re.search(r'\b(19[5-9]\d|20[012]\d)\b', title_blk)
        if ym:
            return int(ym.group(1))
        for pat in self._CITE_YEAR:
            m = pat.search(text[:5000])
            if m:
                y = int(m.group(1))
                if 1950 <= y <= 2025:
                    return y
        m = re.search(r'\b(19[5-9]\d|20[012]\d)\b', text[:3000])
        if m:
            y = int(m.group(1))
            if 1950 <= y <= 2025:
                return y
        return None

    def _sections(self, text: str) -> List[str]:
        found = set()
        for pat in self._SECTIONS:
            for m in pat.finditer(text):
                s = re.sub(r'\s+', ' ', m.group(0).strip())
                if len(s) < 120:
                    found.add(s)
        return list(found)[:30]

    def _verdict(self, tail_text: str) -> str:
        for i, pat in enumerate(self._VERDICT):
            m = pat.search(tail_text)
            if not m:
                continue
            # Patterns 7 (appeal fails) and 8 (no merit) have no capture group
            if m.lastindex is None or i >= 7:
                if i == 7:  return 'dismissed'   # "appeal fails"
                if i == 8:  return 'dismissed'   # "no merit"
                return 'unknown'
            v = m.group(1).lower().strip()
            if 'allow'     in v:  return 'allowed'
            if 'dismiss'   in v:  return 'dismissed'
            if 'partly'    in v:  return 'partly_allowed'
            if 'dispos'    in v:  return 'disposed'
            if 'remand'    in v:  return 'remanded'
            if 'withdraw'  in v:  return 'withdrawn'
            if 'set aside' in v or 'quash' in v:  return 'set_aside'
            if 'affirm'    in v or 'uphold' in v: return 'dismissed'
            if 'modif'     in v:  return 'partly_allowed'
        # All regex patterns failed — ask Mistral Large
        return _mistral_verdict(tail_text)

    def _case_type(self, text: str) -> str:
        m = self._CASE_TYPE.search(text)
        if not m:
            return 'other'
        ct = m.group(1).upper()
        if 'CRIMINAL'      in ct: return 'criminal'
        if 'CIVIL'         in ct: return 'civil'
        if 'WRIT'          in ct: return 'writ'
        if 'SPECIAL LEAVE' in ct or 'SLP' in ct: return 'slp'
        if 'ELECTION'      in ct: return 'election'
        if 'TRANSFER'      in ct: return 'transfer'
        if 'CURATIVE'      in ct: return 'curative'
        if 'REVIEW'        in ct: return 'review'
        return 'other'


_meta_extractor = MetadataExtractor()
print('MetadataExtractor ready')


In [ ]:
# Summarizers
#
# ExtractiveSummarizer (default)
#   Sentence scoring by legal keyword presence + position bonus.
#   Also extracts "Whether..." issues and "held that..." holdings.
#   Speed: ~0 ms/doc, no API, no model.
#
# MistralLLMSummarizer (optional — set LLM_PROVIDER='mistral')
#   Structured JSON output via Mistral Large API.
#   Speed: ~1-2 s/doc, requires MISTRAL_API_KEY.

class ExtractiveSummarizer:
    _KW = [
        'held', 'holding', 'decided', 'ruled', 'ordered', 'directed',
        'dismissed', 'allowed', 'set aside', 'affirmed', 'upheld', 'quashed',
        'constitutional', 'unconstitutional', 'liable', 'convicted', 'acquitted',
        'sentenced', 'compensation', 'principle', 'ratio decidendi',
        'question of law', 'point of law', 'precedent', 'overruled',
        'distinguished', 'followed', 'we find', 'we hold', 'we observe',
        'we are of the view', 'we are of the opinion', 'our view',
        'final conclusions', 'upshot', 'in view of the aforesaid',
    ]

    def summarize(self, text: str, meta: Dict) -> Dict[str, Any]:
        sents = self._sentences(text)
        if len(sents) < 3:
            return {
                'summary': text[:600], 'issues': [],
                'key_holdings': [], 'verdict': meta.get('verdict_type', 'unknown'),
                'legal_significance': '', 'summary_type': 'extractive',
            }
        scores  = self._score(sents)
        n       = min(10, max(4, len(sents) // 16))
        top_idx = sorted(
            sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
        )
        summary = ' '.join(sents[i] for i in top_idx)
        return {
            'summary':           summary[:1800],
            'issues':            self._issues(text)[:4],
            'key_holdings':      self._holdings(text)[:4],
            'verdict':           meta.get('verdict_type', 'unknown'),
            'legal_significance': '',
            'summary_type':      'extractive',
        }

    def _sentences(self, text: str) -> List[str]:
        flat = re.sub(r'\n+', ' ', text)
        sents = re.split(r'(?<=[.!?])\s+(?=[A-Z])', flat)
        return [s.strip() for s in sents if 25 < len(s.strip()) < 600]

    def _score(self, sents: List[str]) -> List[float]:
        total = len(sents)
        scores = []
        for i, s in enumerate(sents):
            sl = s.lower()
            score = sum(1.0 for kw in self._KW if kw in sl)
            if i < 5:               score += 0.5 * (1 - i / 5)
            elif i >= total - 5:    score += 0.5 * (1 - (total - 1 - i) / 5)
            if re.search(r'\(\d{4}\)\s*\d+\s*SCC|AIR\s*\d{4}\s*SC', s): score += 0.4
            if re.search(r'[Ss]ection\s+\d+|[Aa]rticle\s+\d+', s):       score += 0.3
            scores.append(score)
        return scores

    def _issues(self, text: str) -> List[str]:
        pat = re.compile(r'[Ww]hether\s+(.+?)(?:\.|\n)', re.MULTILINE)
        return [
            m.group(1).strip()[:220]
            for m in pat.finditer(text[:6000])
            if len(m.group(1).strip()) > 15
        ][:4]

    def _holdings(self, text: str) -> List[str]:
        pat = re.compile(
            r'(?:[Ww]e|[Tt]his\s+[Cc]ourt|[Tt]he\s+[Cc]ourt)\s+'
            r'(?:hereby\s+)?(?:hold|held|find|observe)s?\s+that\s+(.+?)(?:\.|\n)',
            re.MULTILINE
        )
        return [
            m.group(1).strip()[:220]
            for m in pat.finditer(text)
            if len(m.group(1).strip()) > 15
        ][:4]


class MistralLLMSummarizer:
    _SYS = (
        'You are a legal analyst specialising in Indian Supreme Court judgments. '
        'Extract a structured summary from the provided judgment text. '
        'Respond in JSON only.'
    )
    _TMPL = '''\
Analyse this Supreme Court of India judgment (opening 4000 chars):

{text}

Return ONLY this JSON:
{{
  "summary": "2-3 sentence overview of the case, legal issue, and outcome",
  "issues": ["Legal question 1", "Legal question 2"],
  "key_holdings": ["Key legal principle or holding 1", "Holding 2"],
  "verdict": "allowed|dismissed|partly_allowed|disposed|set_aside|unknown",
  "legal_significance": "Precedential value or significance (1 sentence)"
}}
'''
    _fallback = ExtractiveSummarizer()

    def __init__(self, api_key: str, model: str):
        from mistralai import Mistral
        self.client = Mistral(api_key=api_key)
        self.model  = model

    def summarize(self, text: str, meta: Dict) -> Dict[str, Any]:
        try:
            resp = self.client.chat.complete(
                model=self.model,
                messages=[
                    {'role': 'system', 'content': self._SYS},
                    {'role': 'user',   'content': self._TMPL.format(text=text[:4000])},
                ],
                temperature=0.1,
                max_tokens=700,
                response_format={'type': 'json_object'},
            )
            result = json.loads(resp.choices[0].message.content)
            result['summary_type'] = 'llm'
            return result
        except Exception as exc:
            logger.warning('Mistral summarizer failed: %s', exc)
            return self._fallback.summarize(text, meta)


if LLM_PROVIDER == 'mistral' and MISTRAL_API_KEY:
    _summarizer = MistralLLMSummarizer(api_key=MISTRAL_API_KEY, model=MISTRAL_MODEL)
    print('Summarizer: Mistral Large')
else:
    _summarizer = ExtractiveSummarizer()
    print('Summarizer: Extractive (fast mode)')


In [ ]:
# BGE-M3 Embedder — direct transformers implementation
#
# FlagEmbedding is NOT used here. BGE-M3 is loaded via transformers directly,
# which works on any Python version without C-extension or modeling_* issues.
#
# Encoding matches FlagEmbedding exactly:
#   Dense  : CLS token from last hidden state, L2-normalised
#   Sparse : ReLU(sparse_linear(token_emb)), log(1+w) per-token-id max weight
#            sparse_linear.pt (trained projection head) loaded from BAAI/bge-m3
#
# INDEX_PREFIX applied at index time only (asymmetric) — never at query time.

class BGEM3Direct:
    """BGE-M3 dense + sparse encoder using transformers only.

    API is intentionally identical to FlagEmbedding's BGEM3FlagModel so that
    calling code (LegalEmbedder.encode, cell-16 search test) works unchanged.
    """

    def __init__(self, model_name: str, device: str, use_fp16: bool):
        self._dtype = torch.float16 if (use_fp16 and device == 'cuda') else torch.float32

        print(f'Loading tokenizer: {model_name} ...')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        print('Loading base model weights ...')
        self.model = AutoModel.from_pretrained(model_name, torch_dtype=self._dtype)
        self.model = self.model.to(device)
        self.model.eval()

        hidden_size = self.model.config.hidden_size  # 1024 for bge-m3
        # bias=True to match the saved sparse_linear.pt state dict
        self.sparse_linear = torch.nn.Linear(hidden_size, 1)
        self._load_sparse_linear(model_name, device)

        self.device = device

    def _load_sparse_linear(self, model_name: str, device: str):
        """Load the trained sparse projection head (sparse_linear.pt) from HF."""
        try:
            path = hf_hub_download(model_name, 'sparse_linear.pt')
            try:
                state = torch.load(path, map_location='cpu', weights_only=True)
            except TypeError:
                state = torch.load(path, map_location='cpu')
            self.sparse_linear.load_state_dict(state, strict=False)
            print('Sparse linear projection weights loaded.')
        except Exception as exc:
            print(f'Warning: sparse_linear.pt not loaded ({exc}). Sparse weights approximate.')

        # Keep sparse_linear in the SAME dtype as the base model to avoid
        # "mat1 and mat2 must have the same dtype" on fp16 GPU runs
        self.sparse_linear = self.sparse_linear.to(device=device, dtype=self._dtype)
        self.sparse_linear.eval()

    def encode(
        self,
        texts: List[str],
        batch_size: int = 32,
        max_length: int = 512,
        return_dense: bool = True,
        return_sparse: bool = True,
        return_colbert_vecs: bool = False,
        **kwargs,
    ) -> Dict[str, Any]:
        all_dense: List[torch.Tensor] = []
        all_sparse: List[Dict[int, float]] = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            ).to(self.device)

            with torch.no_grad():
                out = self.model(**inputs)

            # hidden stays in model dtype (fp16 on GPU, fp32 on CPU)
            # so sparse_linear(hidden) has no dtype mismatch
            hidden = out.last_hidden_state          # [B, L, H] — model dtype
            attn   = inputs['attention_mask']       # [B, L]    — int → float below

            if return_dense:
                # Cast to fp32 only after the linear ops are done
                cls = hidden[:, 0, :].float()       # [B, H]
                cls = F.normalize(cls, p=2, dim=-1)
                all_dense.append(cls.cpu())

            if return_sparse:
                # sparse_linear and hidden are the same dtype — no mismatch
                raw_w   = self.sparse_linear(hidden).squeeze(-1)   # [B, L] model dtype
                weights = torch.relu(raw_w).float()                # [B, L] fp32
                weights = weights * attn.float()                   # zero out padding
                log_w   = torch.log(1.0 + weights).cpu()           # [B, L] fp32

                tok_ids = inputs['input_ids'].cpu()                # [B, L]
                for b in range(log_w.shape[0]):
                    sparse: Dict[int, float] = {}
                    for tid, lw in zip(tok_ids[b].tolist(), log_w[b].tolist()):
                        if lw > 0.0 and lw > sparse.get(tid, 0.0):
                            sparse[tid] = lw
                    all_sparse.append(sparse)

        result: Dict[str, Any] = {}
        if return_dense:
            result['dense_vecs'] = torch.cat(all_dense, dim=0)
        if return_sparse:
            result['lexical_weights'] = all_sparse
        return result


class LegalEmbedder:
    _IDX_PREFIX = 'Represent this Indian Supreme Court judgment for legal case retrieval: '

    def __init__(self, model_name: str, device: str, use_fp16: bool):
        # self.model exposes .encode() directly — used by cell-16 search test
        self.model = BGEM3Direct(model_name, device, use_fp16)
        print('BGE-M3 ready.')

    def encode(self, texts: List[str]) -> Tuple[List, List]:
        """Return (dense_list, sparse_list) for a batch of texts."""
        prefixed = [self._IDX_PREFIX + t[:1600] for t in texts]
        out = self.model.encode(
            prefixed,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=512,
            return_dense=True,
            return_sparse=True,
        )
        dense  = out['dense_vecs'].tolist()
        sparse = out.get('lexical_weights', [{} for _ in texts])
        return dense, sparse


def build_embed_text(meta: Dict, summ: Dict) -> str:
    """Compose the text that gets embedded."""
    parts: List[str] = []
    if meta.get('case_title'):      parts.append(meta['case_title'])
    if meta.get('petitioner'):      parts.append(f"Petitioner: {meta['petitioner']}")
    if meta.get('respondent'):      parts.append(f"Respondent: {meta['respondent']}")
    if summ.get('summary'):         parts.append(summ['summary'])
    if summ.get('issues'):          parts.append('Issues: ' + '; '.join(summ['issues'][:3]))
    if summ.get('key_holdings'):    parts.append('Holdings: ' + '; '.join(summ['key_holdings'][:3]))
    if meta.get('legal_sections'):  parts.append('Sections: ' + ', '.join(meta['legal_sections'][:12]))
    v = meta.get('verdict_type', '')
    if v and v != 'unknown':        parts.append(f'Verdict: {v}')
    return ' | '.join(parts)[:2200]


# First run downloads ~2.3 GB from HuggingFace (cached after that in /kaggle/working)
_embedder = LegalEmbedder(EMBEDDING_MODEL, DEVICE, USE_FP16)

In [ ]:
# Checkpoint Manager
# Persists progress to disk so the pipeline can be interrupted and resumed.

class CheckpointManager:
    def __init__(self, path: str):
        self._path = Path(path)
        self.processed: set = set()
        self.failed:    set = set()
        self.stats = {'parsed': 0, 'ingested': 0, 'failed': 0}
        self._load()

    def _load(self):
        if self._path.exists():
            data = json.loads(self._path.read_text())
            self.processed = set(data.get('processed', []))
            self.failed    = set(data.get('failed', []))
            self.stats     = data.get('stats', self.stats)
            print(f'Checkpoint: {len(self.processed):,} done, {len(self.failed):,} failed — resuming')
        else:
            print('No checkpoint found — fresh start')

    def save(self):
        self._path.write_text(json.dumps({
            'processed': sorted(self.processed),
            'failed':    sorted(self.failed),
            'stats':     self.stats,
            'saved_at':  time.strftime('%Y-%m-%dT%H:%M:%S'),
        }, indent=2))

    def is_done(self, name: str) -> bool:
        return name in self.processed or name in self.failed

    def mark_parsed(self, name: str):
        self.processed.add(name)
        self.stats['parsed'] += 1

    def mark_failed(self, name: str, reason: str = ''):
        self.failed.add(name)
        self.stats['failed'] += 1
        with open(ERROR_LOG, 'a') as f:
            f.write(json.dumps({'file': name, 'reason': str(reason)[:300]}) + '\n')

    def mark_ingested(self, n: int):
        self.stats['ingested'] += n

    def reset(self):
        """Wipe checkpoint and error log for a clean run."""
        if self._path.exists():
            self._path.unlink()
        error_path = Path(ERROR_LOG)
        if error_path.exists():
            error_path.unlink()
        self.processed = set()
        self.failed    = set()
        self.stats     = {'parsed': 0, 'ingested': 0, 'failed': 0}
        print('Checkpoint cleared — all files will be re-processed')


# FRESH_START clears checkpoint + error log before initialising
if FRESH_START:
    # Wipe files directly so CheckpointManager.__init__ sees a clean slate
    for _p in [Path(CHECKPOINT_FILE), Path(ERROR_LOG)]:
        if _p.exists():
            _p.unlink()
    print('FRESH_START=True — checkpoint and error log cleared')

_checkpoint = CheckpointManager(CHECKPOINT_FILE)
print('CheckpointManager ready')


In [ ]:
# Single Document Processor
# Called from thread pool — parses, extracts, summarises one PDF.
#
# Drop policy (conservative — ingest as much as possible):
#   • PDF parse completely failed AND produced no URL → skip (nothing to index)
#   • PDF is readable but text < 100 chars AND no URL → skip (empty scanned image)
#   • All other cases → ingest.  verdict_type defaults to 'unknown' when not found.
#     The Indian Kanoon URL is the recovery path — users can always open the source.

def process_pdf(pdf_path: str, year_from_path: int = 0) -> Optional[Dict[str, Any]]:
    """Process a single PDF judgment.

    Args:
        pdf_path:       Absolute path to the PDF file.
        year_from_path: Year extracted from the parent directory name (e.g. 2003).
                        When non-zero, overrides any year parsed from document text —
                        directory structure is far more reliable than text regex.
    """
    file_name = Path(pdf_path).name

    # ── Hard gate: skip anything before START_YEAR ─────────────────────
    if year_from_path and year_from_path < START_YEAR:
        logger.debug('Skipping %s — year %d < START_YEAR %d', file_name, year_from_path, START_YEAR)
        return None

    try:
        # 1 — Parse PDF (two-pass: URL from footer, then clean body text)
        parsed = _parser.parse(pdf_path)

        ik_url = parsed.get('ik_url') if parsed else None
        text   = (parsed.get('clean_text') or '') if parsed else ''

        # Drop ONLY if truly unreadable AND we have no URL to fall back on.
        # A document with a URL but poor OCR is still valuable — the user can
        # open the source via indian_kanoon_url.
        if not text and not ik_url:
            _checkpoint.mark_failed(file_name, 'parse_failed_no_url')
            return None

        if len(text) < 100 and not ik_url:
            _checkpoint.mark_failed(file_name, 'too_short_no_url')
            return None

        structured = (parsed.get('structured') or {}) if parsed else {}

        # 2 — Extract structured metadata
        meta = _meta_extractor.extract(text, structured, file_name)
        meta['indian_kanoon_url'] = ik_url or ''
        meta['has_url']           = ik_url is not None

        # 2b — Override year with directory-derived value (more reliable than text regex)
        if year_from_path:
            meta['year'] = year_from_path

        # 2c — Final year sanity-check
        doc_year = meta.get('year')
        if doc_year and doc_year < START_YEAR:
            logger.debug('Skipping %s — extracted year %d < START_YEAR %d', file_name, doc_year, START_YEAR)
            return None

        # 3 — Generate summary (extractive; works on short text too)
        summ = _summarizer.summarize(text, meta)

        # 4 — Build embedding text
        embed_text = build_embed_text(meta, summ)

        _checkpoint.mark_parsed(file_name)

        return {
            'file_name':     file_name,
            'page_count':    (parsed.get('page_count') or 0) if parsed else 0,
            'text_preview':  text[:PREVIEW_CHARS],
            'full_text':     text[:MAX_TEXT_STORE],
            'embed_text':    embed_text,
            'meta':          meta,
            'summ':          summ,
        }

    except Exception as exc:
        _checkpoint.mark_failed(file_name, str(exc)[:250])
        logger.error('process_pdf %s: %s', file_name, exc)
        return None


print('process_pdf() ready')


In [ ]:
# Batch Qdrant Ingestion
#
# Payload schema:
#
# Identity     : file_name, source, page_count
# Case         : case_title, case_number, case_type
#                petitioner, respondent, author, judges
#                citation, all_citations, date, year
#                legal_sections, verdict_type
# URL          : indian_kanoon_url, has_url
# Summary      : summary, issues, key_holdings, verdict
#                legal_significance, summary_type
# Text         : text_preview  ← first 1800 chars (snippet display)
#                text_tail     ← last 2500 chars  (verdict re-processing)

def ingest_batch(docs: List[Dict]) -> int:
    if not docs:
        return 0

    dense_vecs, sparse_dicts = _embedder.encode([d['embed_text'] for d in docs])

    points: List[PointStruct] = []
    for i, doc in enumerate(docs):
        meta = doc['meta']
        summ = doc['summ']
        full  = doc.get('full_text', '')

        vectors: Dict[str, Any] = {'dense': dense_vecs[i]}
        if USE_SPARSE and sparse_dicts[i]:
            vectors['sparse'] = SparseVector(
                indices=list(sparse_dicts[i].keys()),
                values=list(sparse_dicts[i].values()),
            )

        payload = {
            # ── Identity ─────────────────────────────────────────────
            'file_name':    doc['file_name'],
            'source':       'indian_kanoon',
            'page_count':   doc.get('page_count', 0),

            # ── Case details ─────────────────────────────────────────
            'case_title':   meta.get('case_title', ''),
            'case_number':  meta.get('case_number', ''),
            'case_type':    meta.get('case_type', 'other'),
            'petitioner':   meta.get('petitioner', ''),
            'respondent':   meta.get('respondent', ''),
            'author':       meta.get('author', ''),
            'judges':       meta.get('judges', []),
            'citation':     meta.get('primary_citation', ''),
            'all_citations': meta.get('citations', []),
            'date':         meta.get('date', ''),
            'year':         meta.get('year'),
            'legal_sections': meta.get('legal_sections', []),
            'verdict_type': meta.get('verdict_type', 'unknown'),

            # ── CRITICAL — link to full document ─────────────────────
            'indian_kanoon_url': meta.get('indian_kanoon_url', ''),
            'has_url':      meta.get('has_url', False),

            # ── Summary ──────────────────────────────────────────────
            'summary':      summ.get('summary', ''),
            'issues':       summ.get('issues', []),
            'key_holdings': summ.get('key_holdings', []),
            'verdict':      summ.get('verdict', 'unknown'),
            'legal_significance': summ.get('legal_significance', ''),
            'summary_type': summ.get('summary_type', 'extractive'),

            # ── Text ─────────────────────────────────────────────────
            'text_preview': doc.get('text_preview', ''),
            # Last 2500 chars stored for verdict re-processing without re-parsing PDFs
            'text_tail':    full[-2500:] if len(full) > 2500 else full,
        }

        point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, doc['file_name']))
        points.append(PointStruct(id=point_id, vector=vectors, payload=payload))

    qdrant.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    _checkpoint.mark_ingested(len(points))
    return len(points)


print('ingest_batch() ready')

## ▶️ Run Ingestion

**Safe to interrupt** — progress is checkpointed every batch.  
Re-running the cell will skip already-processed files.

| Stage | Estimate (T4 GPU, 26K docs) |
|-------|-----------------------------|
| PDF parsing (4 threads) | ~1.8 h |
| BGE-M3 embedding (GPU)  | ~15 min |
| Qdrant upsert           | ~10 min |
| **Total**               | **~2–3 h** |

In [ ]:
def run_ingestion():
    """Discover PDFs from year subdirectories >= START_YEAR and ingest them."""
    data_root = Path(JUDGMENTS_DIR)

    # ── 1. Collect year directories ────────────────────────────────────────
    year_dirs = sorted(
        [
            d for d in data_root.iterdir()
            if d.is_dir() and d.name.isdigit() and int(d.name) >= START_YEAR
        ],
        key=lambda d: int(d.name),
    )
    if not year_dirs:
        print(f'ERROR: No year directories >= {START_YEAR} found under {JUDGMENTS_DIR}')
        print('  → Check that the dataset is added as a notebook input')
        return

    print(f'Year range       : {year_dirs[0].name} – {year_dirs[-1].name}  ({len(year_dirs)} dirs)')

    # ── 2. Collect PDF paths (uppercase .PDF first, then lowercase .pdf) ──
    pdf_files: List[Path] = []
    for d in year_dirs:
        pdf_files.extend(sorted(d.glob('*.PDF')))
        pdf_files.extend(sorted(d.glob('*.pdf')))

    if not pdf_files:
        print(f'ERROR: No PDFs found in year directories >= {START_YEAR}')
        print('  → Confirm the dataset path and that PDFs exist')
        return

    print(f'PDFs found       : {len(pdf_files):,}')
    if MAX_DOCS:
        pdf_files = pdf_files[:MAX_DOCS]
        print(f'Capped at        : {MAX_DOCS:,}')

    remaining = [p for p in pdf_files if not _checkpoint.is_done(p.name)]
    print(f'Already done     : {len(pdf_files) - len(remaining):,}')
    print(f'To process       : {len(remaining):,}')

    if not remaining:
        print('Nothing left — ingestion complete.')
        return

    # ── 3. Process in parallel, ingest in batches ──────────────────────────
    batch: List[Dict] = []
    t0 = time.time()

    with tqdm(total=len(remaining), desc='Ingesting SC Judgments', unit='doc') as pbar:
        with ThreadPoolExecutor(max_workers=WORKER_THREADS) as pool:
            # Pass year from directory name — more reliable than text-regex extraction
            futures = {
                pool.submit(process_pdf, str(p), int(p.parent.name)): p
                for p in remaining
            }

            for future in as_completed(futures):
                result = future.result()
                pbar.update(1)

                if result:
                    batch.append(result)

                if len(batch) >= BATCH_SIZE:
                    try:
                        ingest_batch(batch)
                    except Exception as exc:
                        logger.error('Batch error: %s', exc)
                    batch = []
                    _checkpoint.save()

                    elapsed = time.time() - t0
                    rate    = _checkpoint.stats['ingested'] / max(elapsed, 1) * 3600
                    pbar.set_postfix({
                        'ingested': f"{_checkpoint.stats['ingested']:,}",
                        'failed':   _checkpoint.stats['failed'],
                        'docs/h':   f'{rate:.0f}',
                    })

                if LLM_PROVIDER != 'none':
                    time.sleep(LLM_RPM_DELAY)

    # ── 4. Flush remaining batch ───────────────────────────────────────────
    if batch:
        try:
            ingest_batch(batch)
        except Exception as exc:
            logger.error('Final batch: %s', exc)

    _checkpoint.save()

    elapsed  = time.time() - t0
    final_ct = qdrant.count(COLLECTION_NAME).count
    print('\n' + '=' * 55)
    print('  INGESTION COMPLETE')
    print('=' * 55)
    print(f'  Years     : {START_YEAR} – present')
    print(f'  Parsed    : {_checkpoint.stats["parsed"]:>8,}')
    print(f'  Ingested  : {_checkpoint.stats["ingested"]:>8,}')
    print(f'  Failed    : {_checkpoint.stats["failed"]:>8,}')
    print(f'  Time      : {elapsed/3600:.2f} h')
    print(f'  Qdrant    : {final_ct:>8,} points in \'{COLLECTION_NAME}\'')
    print('=' * 55)


run_ingestion()

In [ ]:
# Post-Ingestion Statistics

info  = qdrant.get_collection(COLLECTION_NAME)
count = qdrant.count(COLLECTION_NAME).count

print(f'Collection  : {COLLECTION_NAME}')
print(f'Points      : {count:,}')
print(f'Status      : {info.status}')

# disk_data_size / ram_data_size were removed in qdrant-client >= 1.9
# Try the old attributes, then the nested config path, then skip gracefully
disk_mb = getattr(info, 'disk_data_size', None)
ram_mb  = getattr(info, 'ram_data_size',  None)
if disk_mb is not None:
    print(f'Disk usage  : {disk_mb / 1e6:.1f} MB')
    print(f'RAM usage   : {ram_mb  / 1e6:.1f} MB')
else:
    print('Disk / RAM  : (not available in this qdrant-client version)')

print('\nYear distribution (only years >= START_YEAR ingested):')
for start, end in [(2000, 2004), (2005, 2009), (2010, 2014), (2015, 2019), (2020, 2025)]:
    n = qdrant.count(
        COLLECTION_NAME,
        count_filter=Filter(must=[
            FieldCondition(key='year', range=Range(gte=start, lte=end))
        ]),
    ).count
    bar = '#' * (n * 40 // max(count, 1))
    print(f'  {start}-{end} : {n:>5,}  {bar}')

with_url = qdrant.count(
    COLLECTION_NAME,
    count_filter=Filter(must=[
        FieldCondition(key='has_url', match=MatchValue(value=True))
    ]),
).count
print(f'\nWith Indian Kanoon URL : {with_url:,} / {count:,} ({with_url / max(count, 1) * 100:.1f}%)')

print('\nVerdict breakdown:')
for v in ['allowed', 'dismissed', 'partly_allowed', 'set_aside', 'disposed', 'unknown']:
    n = qdrant.count(
        COLLECTION_NAME,
        count_filter=Filter(must=[
            FieldCondition(key='verdict_type', match=MatchValue(value=v))
        ]),
    ).count
    if n > 0:
        print(f'  {v:<18} : {n:,}')

print('\nCase type breakdown:')
for ct in ['criminal', 'civil', 'writ', 'slp', 'election', 'transfer', 'curative', 'review', 'other']:
    n = qdrant.count(
        COLLECTION_NAME,
        count_filter=Filter(must=[
            FieldCondition(key='case_type', match=MatchValue(value=ct))
        ]),
    ).count
    if n > 0:
        print(f'  {ct:<18} : {n:,}')

In [ ]:
# Targeted Re-parse — fix remaining verdict_type='unknown' documents
#
# Strategy (cheapest first):
#   Stage 1b — text_tail  : regex on last 2500 chars stored in payload
#   Stage 3   — PDF re-parse : read actual PDF, extract last 20%, regex only
#
# IMPORTANT: Mistral fallback is DISABLED here intentionally.
# It already ran during ingestion (in process_pdf → _verdict()).
# Calling it again on the same tail text would produce the same result
# and make 1,500+ sequential API calls.  Regex only in this cell.

import concurrent.futures
from pathlib import Path
from qdrant_client.models import PointIdsList

# ── Disable Mistral for the duration of this cell ─────────────────────
_mistral_verdict_orig = _mistral_verdict          # save original
_mistral_verdict      = lambda _tail: 'unknown'   # no-op in this cell

_SCROLL_BATCH    = 256
_REPARSE_WORKERS = 4

# ── Stage 0: count unknowns before we start ───────────────────────────
_n_unknown_before = qdrant.count(
    COLLECTION_NAME,
    count_filter=Filter(must=[
        FieldCondition(key='verdict_type', match=MatchValue(value='unknown'))
    ]),
).count
print(f'Unknown-verdict points before re-parse : {_n_unknown_before:,}')
if _n_unknown_before == 0:
    print('Nothing to do.')
    _mistral_verdict = _mistral_verdict_orig
    raise SystemExit(0)

# ── Stage 1: collect all unknown-verdict points ────────────────────────
print('\nStage 1 — collecting unknown-verdict points ...')
unknown_pts = []
offset = None
while True:
    pts, next_offset = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=Filter(must=[
            FieldCondition(key='verdict_type', match=MatchValue(value='unknown'))
        ]),
        limit=_SCROLL_BATCH,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    unknown_pts.extend(pts)
    if next_offset is None:
        break
    offset = next_offset
print(f'  Collected {len(unknown_pts):,} points')

# ── Stage 1b: try stored text_tail (regex only, zero API calls) ────────
print('\nStage 1b — trying stored text_tail (regex only) ...')
_ext     = MetadataExtractor()
need_pdf = []
tail_resolved = 0

for pt in unknown_pts:
    tail = pt.payload.get('text_tail', '')
    if tail:
        v = _ext._verdict(tail)        # Mistral is patched out — regex only
        if v != 'unknown':
            qdrant.set_payload(
                collection_name=COLLECTION_NAME,
                payload={'verdict_type': v},
                points=PointIdsList(points=[pt.id]),
            )
            tail_resolved += 1
            continue
    need_pdf.append(pt)

print(f'  Resolved from text_tail : {tail_resolved:,}')
print(f'  Need PDF re-parse       : {len(need_pdf):,}')

# ── Stage 2: build file_name → Path index ─────────────────────────────
print('\nStage 2 — building file_name → path index ...')
name_to_path: Dict[str, Path] = {}
year_dirs = sorted(
    [d for d in Path(JUDGMENTS_DIR).iterdir()
     if d.is_dir() and d.name.isdigit() and int(d.name) >= START_YEAR],
    key=lambda d: int(d.name),
)
for d in year_dirs:
    for p in d.iterdir():
        if p.suffix.lower() == '.pdf':
            name_to_path[p.name] = p

found_pdfs   = sum(1 for pt in need_pdf if pt.payload.get('file_name', '') in name_to_path)
missing_pdfs = len(need_pdf) - found_pdfs
print(f'  Year dirs scanned : {len(year_dirs)}')
print(f'  Total PDFs mapped : {len(name_to_path):,}')
print(f'  Matched to points : {found_pdfs:,}  (missing on disk: {missing_pdfs:,})')

# ── Stage 3: parallel PDF re-parse (regex only) ────────────────────────
if need_pdf:
    print(f'\nStage 3 — re-parsing {len(need_pdf):,} PDFs (regex only) ...')
    _psr = _parser   # SCJudgmentParser instance created in cell-05

    def _reparse_one(pt):
        file_name = pt.payload.get('file_name', '')
        pdf_path  = name_to_path.get(file_name)
        if pdf_path is None:
            return pt.id, 'unknown', ''
        try:
            doc = _psr.parse(str(pdf_path))
            if not doc:
                return pt.id, 'unknown', ''
            full = doc.get('full_text', '') or ''
            if not full:
                return pt.id, 'unknown', ''
            tail_for_verdict = full[int(len(full) * 0.80):]
            tail_for_store   = full[-2500:] if len(full) > 2500 else full
            verdict = _ext._verdict(tail_for_verdict)   # regex only (Mistral patched out)
            return pt.id, verdict, tail_for_store
        except Exception:
            return pt.id, 'unknown', ''

    pdf_resolved      = 0
    pdf_still_unknown = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=_REPARSE_WORKERS) as pool:
        futs = {pool.submit(_reparse_one, pt): pt for pt in need_pdf}
        for i, fut in enumerate(concurrent.futures.as_completed(futs), 1):
            point_id, verdict, text_tail = fut.result()
            payload_patch = {'verdict_type': verdict}
            if text_tail:
                payload_patch['text_tail'] = text_tail
            qdrant.set_payload(
                collection_name=COLLECTION_NAME,
                payload=payload_patch,
                points=PointIdsList(points=[point_id]),
            )
            if verdict != 'unknown':
                pdf_resolved += 1
            else:
                pdf_still_unknown += 1
            if i % 500 == 0 or i == len(need_pdf):
                pct = i / len(need_pdf) * 100
                print(f'  [{i:>5,}/{len(need_pdf):,}]  {pct:5.1f}%'
                      f'  resolved {pdf_resolved:,}  still-unknown {pdf_still_unknown:,}')
else:
    pdf_resolved      = 0
    pdf_still_unknown = 0
    print('\nStage 3 skipped.')

# ── Restore Mistral for the rest of the notebook ──────────────────────
_mistral_verdict = _mistral_verdict_orig

# ── Summary ────────────────────────────────────────────────────────────
total_resolved = tail_resolved + pdf_resolved
print(f'\n{"─"*54}')
print(f'Re-parse summary')
print(f'{"─"*54}')
print(f'  Started with unknown   : {_n_unknown_before:,}')
print(f'  Resolved via text_tail : {tail_resolved:,}')
print(f'  Resolved via PDF parse : {pdf_resolved:,}')
print(f'  Total resolved         : {total_resolved:,}')
print(f'  Still unknown          : {pdf_still_unknown + missing_pdfs:,}')

# ── Final verdict breakdown ────────────────────────────────────────────
print('\nFinal verdict distribution:')
_total = qdrant.count(COLLECTION_NAME).count
for _v in ['allowed', 'dismissed', 'partly_allowed', 'set_aside',
           'disposed', 'remanded', 'withdrawn', 'unknown']:
    _n = qdrant.count(
        COLLECTION_NAME,
        count_filter=Filter(must=[
            FieldCondition(key='verdict_type', match=MatchValue(value=_v))
        ]),
    ).count
    if _n > 0:
        _bar = '█' * int(_n / _total * 40)
        print(f'  {_v:<18} : {_n:>6,}  ({_n/_total*100:5.1f}%)  {_bar}')


In [ ]:
# Verification: Hybrid Semantic Search Test
#
# Uses Reciprocal Rank Fusion (RRF) to merge dense + sparse results — same strategy
# as the main Neethi RAG pipeline (backend/rag/).
#
# dense  → semantic / conceptual similarity (BGE-M3 1024-dim)
# sparse → keyword / BM25 term match       (BGE-M3 lexical weights)
# RRF    → merges both ranked lists without requiring score calibration

from qdrant_client.models import Prefetch, FusionQuery, Fusion

def search(
    query: str,
    top_k: int = 5,
    year_from: int = None,
    year_to: int = None,
    verdict: str = None,
):
    """Hybrid search (dense + sparse RRF) over the indian_kanoon collection."""
    # Encode query — no prefix (asymmetric: prefix only at index time)
    q_out = _embedder.model.encode(
        [query],
        batch_size=1,
        max_length=256,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    q_dense = q_out['dense_vecs'][0].tolist()
    q_sparse_raw = q_out['lexical_weights'][0]
    q_sparse = SparseVector(
        indices=[int(k) for k in q_sparse_raw.keys()],
        values=[float(v) for v in q_sparse_raw.values()],
    )

    # Optional payload filter
    must = []
    if year_from or year_to:
        must.append(FieldCondition(
            key='year',
            range=Range(gte=year_from or 1950, lte=year_to or 2025),
        ))
    if verdict:
        must.append(FieldCondition(key='verdict_type', match=MatchValue(value=verdict)))

    filt = Filter(must=must) if must else None

    if USE_SPARSE:
        # ── Hybrid: dense + sparse fused with RRF ──────────────────────────
        results = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=[
                Prefetch(
                    query=q_dense,
                    using='dense',
                    limit=top_k * 3,     # over-fetch for RRF headroom
                    filter=filt,
                ),
                Prefetch(
                    query=q_sparse,
                    using='sparse',
                    limit=top_k * 3,
                    filter=filt,
                ),
            ],
            query=FusionQuery(fusion=Fusion.RRF),
            limit=top_k,
            with_payload=True,
        ).points
    else:
        # ── Dense only (fallback when USE_SPARSE=False) ────────────────────
        results = qdrant.search(
            collection_name=COLLECTION_NAME,
            query_vector=('dense', q_dense),
            limit=top_k,
            with_payload=True,
            score_threshold=0.3,
            query_filter=filt,
        )

    mode = 'dense+sparse RRF' if USE_SPARSE else 'dense only'
    print(f'  [{mode}]  {len(results)} results')
    for i, r in enumerate(results, 1):
        p = r.payload
        print(f'\n{"-"*65}')
        print(f'#{i}  Score  : {r.score:.4f}')
        print(f'    Title  : {p.get("case_title","N/A")[:80]}')
        print(f'    Parties: {p.get("petitioner","?")[:40]} v. {p.get("respondent","?")[:40]}')
        print(f'    Judges : {p.get("author","?")}  |  Bench: {p.get("judges",[])}')
        print(f'    Cite   : {p.get("citation","N/A")}  ({p.get("year","?")})  |  Verdict: {p.get("verdict_type","?")}')
        print(f'    URL    : {p.get("indian_kanoon_url") or "NOT EXTRACTED"}')
        print(f'    Summary: {(p.get("summary") or "")[:220]}…')
        if p.get('key_holdings'):
            for h in p['key_holdings'][:2]:
                print(f'    > {h[:110]}')
        if p.get('legal_sections'):
            print(f'    Sections: {p["legal_sections"][:4]}')


print('=' * 65)
print('TEST 1: Section 138 NI Act + IBC moratorium')
print('=' * 65)
search('Section 138 NI Act proceedings after IBC resolution plan moratorium corporate debtor')

print('\n\n' + '=' * 65)
print('TEST 2: Anticipatory bail criminal procedure')
print('=' * 65)
search('anticipatory bail right of accused criminal procedure')

print('\n\n' + '=' * 65)
print('TEST 3: Right to property Article 300A')
print('=' * 65)
search('right to property Article 300A land acquisition compensation')